In [1]:
import pandas as pd 
import json
import mne
import numpy as np
import mne
import os
from itertools import product
import glob

from keras import ops

print("it started")
import os
import random
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import numpy as np
import tensorflow as tf
import torch

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# ---------------------------------------------------------------------------
# ORIGINAL IMPORTS & SETUP
# ---------------------------------------------------------------------------
import json
import uuid
import pandas as pd
import matplotlib.pyplot as plt

# Scipy & MNE
import mne
from scipy.signal import stft, welch
from scipy.stats import entropy, norm
from sklearn.model_selection import KFold, train_test_split

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
from tensorflow.keras import layers, models, Model, callbacks

print(f"Reproducibility settings locked with SEED: {SEED}")

# GPU Check
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

it started
Reproducibility settings locked with SEED: 42
TensorFlow GPU Accelerated Backend Active.
CUDA GPU Accelerated Backend Active: Tesla T4


In [2]:
# Path to the participants TSV file
file_path = "/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /participants.tsv"

# Read the tab-separated file
df = pd.read_csv(file_path, sep='\t')

# Display the first few rows
df.head()

,participant_id,subject_id,group,updrs_part_iii,updrs_total,moca,age,sex,disease_duration,ledd,pigd_score,td_score,ctt
0,sub-001,HC0001,HC,0.0,0.0,30.0,42.0,M,NaN,NaN,NaN,NaN,NaN
1,sub-002,HC0003,HC,2.0,3.0,27.0,60.0,M,NaN,NaN,NaN,NaN,66.0
2,sub-003,HC0004,HC,0.0,1.0,27.0,60.0,F,NaN,NaN,NaN,NaN,63.0
3,sub-004,HC0005,HC,1.0,1.0,25.0,72.0,M,NaN,NaN,NaN,NaN,116.0
4,sub-005,HC0006,HC,NaN,NaN,NaN,47.0,M,NaN,NaN,NaN,NaN,NaN


In [3]:
sub_condition = df.iloc[:,2].values
print(sub_condition[0:5])

['HC' 'HC' 'HC' 'HC' 'HC']


In [4]:
nan_counts = df.isna().sum()
print(nan_counts)

participant_id       0
subject_id           0
group                0
updrs_part_iii       5
updrs_total          5
moca                 4
age                  0
sex                  0
disease_duration    28
ledd                29
pigd_score          31
td_score            31
ctt                  9
dtype: int64


In [5]:
print(df.shape)

(144, 13)


In [6]:
missing_ids = []

for i in range(1, 145):
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"
    
    # Check if file exists; if not, store or print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

In [7]:
missing_ids = []

for i in range(1, 145):
    # Format i with 3-digit zero-padding (e.g., 001, 002, ..., 144)
    sub_id = f"{i:03d}"
    file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-walk_eeg.set"
    
    # Check if the file does NOT exist and print i
    if not os.path.exists(file_path):
        print(i)
        missing_ids.append(i)

1
5
16
20
25
36
43
84
100
120
126


In [8]:


# Format participant ID as 3-digit zero-padded string ('001')
sub_id = f"{1:03d}"
file_path = f"/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 /sub-{sub_id}/eeg/sub-{sub_id}_task-rest_eeg.set"

# Load the file into memory
raw = mne.io.read_raw_eeglab(file_path, preload=True)

# Extract raw numerical array: shape is (Channels, Length)
signal = raw.get_data()

print("Signal shape (C, L):", signal.shape)

Signal shape (C, L): (65, 60964)


/tmp/ipykernel_465/1231990765.py:6: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True)


In [9]:
sfreq = raw.info['sfreq']

print(f"Sampling Frequency: {sfreq} Hz")

Sampling Frequency: 250.0 Hz


In [10]:
import mne
import numpy as np

def get_eeg_signal(
    sub_id, 
    task="walk", 
    band="full",
    target_sfreq=256,
    duration=2.0, 
    notch_freq=50.0, 
    reject_threshold=0.00028,
    noise_db=None,
    base_dir="/kaggle/input/datasets/adithyarajnarayanan/eeg-pd-dataset-part-3/EEG dataset part 3 "
):
    """
    Loads, cleans, filters by frequency band, resamples, segments EEG data across all channels,
    and optionally adds Gaussian noise based on a target Signal-to-Noise Ratio (SNR in dB).

    Parameters:
    -----------
    noise_db : float or None
        Target Signal-to-Noise Ratio (SNR) in dB.
        For example, noise_db=20 adds noise such that SNR = 20 dB.
        If None, no noise is added.

    Returns:
    --------
    float32 NumPy array of shape (N_epochs, Channels, Time).
    """
    # 1. Map band names to frequency limits
    band_limits = {
        'full':  (1.0, 45.0),
        'delta': (1.0, 4.0),
        'theta': (4.0, 8.0),
        'alpha': (8.0, 12.0),
        'beta':  (12.0, 30.0),
        'gamma': (30.0, 45.0)
    }
    
    if band.lower() not in band_limits:
        raise ValueError(f"Invalid band '{band}'. Choose from: {list(band_limits.keys())}")
        
    l_freq, h_freq = band_limits[band.lower()]

    # 2. Format subject ID
    if isinstance(sub_id, int):
        sub_str = f"{sub_id:03d}"
    else:
        sub_str = str(sub_id).zfill(3)

    file_path = f"{base_dir}/sub-{sub_str}/eeg/sub-{sub_str}_task-{task}_eeg.set"

    # 3. Load continuous file
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

    # 4. Fix channel types & keep full EEG channels
    channel_type_mapping = {
        'EOG1': 'eog', 'EOG2': 'eog', 'EOG3': 'eog', 'EOG4': 'eog', 'VREF': 'misc'
    }
    existing_mapping = {ch: t for ch, t in channel_type_mapping.items() if ch in raw.ch_names}
    if existing_mapping:
        raw.set_channel_types(existing_mapping)

    raw.pick_types(eeg=True, eog=False, misc=False)

    # 5. PREPROCESSING
    # A. Bandpass filter for selected band
    raw.filter(l_freq=l_freq, h_freq=h_freq, fir_design='firwin', verbose=False)

    # B. Notch Filter
    if notch_freq is not None and h_freq >= notch_freq:
        raw.notch_filter(freqs=notch_freq, verbose=False)

    # C. Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average', projection=False, verbose=False)

    # 6. Resample to target frequency
    raw.resample(sfreq=target_sfreq, verbose=False)

    # 7. SEGMENTATION (Epoching full continuous signal)
    events = mne.make_fixed_length_events(raw, duration=duration)
    reject_criteria = dict(eeg=reject_threshold) if reject_threshold is not None else None

    epochs = mne.Epochs(
        raw, 
        events=events, 
        tmin=0, 
        tmax=duration - (1 / target_sfreq), 
        baseline=None, 
        reject=reject_criteria,
        preload=True, 
        verbose=False
    )

    # Extract array and cast to float32
    signal = epochs.get_data().astype(np.float32)

    # 8. ADD GAUSSIAN NOISE (SNR in dB)
    if noise_db is not None:
        # Calculate signal power across all dimensions
        signal_power = np.mean(signal ** 2)
        
        if signal_power > 0:
            # Convert target SNR from dB to linear scale: SNR_linear = 10^(SNR_dB / 10)
            snr_linear = 10.0 ** (noise_db / 10.0)
            
            # Calculate required noise power: Noise_Power = Signal_Power / SNR_linear
            noise_power = signal_power / snr_linear
            noise_std = np.sqrt(noise_power)
            
            # Generate zero-mean Gaussian noise with calculated std
            gaussian_noise = np.random.normal(loc=0.0, scale=noise_std, size=signal.shape).astype(np.float32)
            
            # Add noise to signal
            signal = signal + gaussian_noise

    return signal

In [11]:
non_walk_ids = [1,5,16,20,25,36,43,84,100,120,126]

In [12]:
rest_ids = [i for i in range(1,145)]
walk_ids = [i for i in range(1,145) if i not in non_walk_ids]

In [13]:
def get_data(task, band, rest_ids, walk_ids):
    X_hc = []
    X_pd = []
    
    # Select subject list based on task
    sub_ids = rest_ids if task == "rest" else walk_ids
    
    for sub_id in sub_ids:
        try:
            # Extract signal for the current subject
            eeg_signal = get_eeg_signal(sub_id=sub_id, task=task, band=band)
            
            # Split into HC (< 29) or PD (>= 29)
            if sub_id < 29:
                X_hc.append(eeg_signal)
            else:
                X_pd.append(eeg_signal)
                
        except Exception as e:
            print(f"Skipping Subject {sub_id} ({task}, {band}) due to error: {e}")
            
    return X_hc, X_pd


In [14]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1):
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]
    
    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)
    
    if total_c0 == total_c1:
        return np.concatenate(X_list_c0, axis=0), np.concatenate(X_list_c1, axis=0)

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active
        
        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break
            
        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0
                
                available = maj_counts[i] - allocations[i]
                take = min(share, available)
                
                allocations[i] += take
                remaining_target -= take
                
                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])
            
    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [15]:
def scale_data(X_list):
    scaled = []
    for sub in X_list:
        flat = sub.reshape(-1, sub.shape[-1])
        mu = np.mean(flat, axis=0)
        std = np.std(flat, axis=0) + 1e-8
        scaled.append((sub - mu) / std)
    return scaled

In [16]:
class RearrangeToSeq(layers.Layer):
    """Reshapes (Batch, 1, Time, Filters) -> (Batch, Time, Filters) for Transformer inputs."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, x):
        return ops.squeeze(x, axis=1)


class MultiHeadSelfAttention(layers.Layer):
    """Multi-Head Self-Attention using scaled dot-product compatible with Keras 3."""
    def __init__(self, emb_size=40, num_heads=10, dropout=0.5, **kwargs):
        super().__init__(**kwargs)
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.head_dim = emb_size // num_heads

        self.q_dense = layers.Dense(emb_size)
        self.k_dense = layers.Dense(emb_size)
        self.v_dense = layers.Dense(emb_size)
        self.out_dense = layers.Dense(emb_size)
        self.att_drop = layers.Dropout(dropout)

    def split_heads(self, x, batch_size):
        x = ops.reshape(x, (batch_size, -1, self.num_heads, self.head_dim))
        return ops.transpose(x, axes=[0, 2, 1, 3])

    def call(self, x, training=False):
        batch_size = ops.shape(x)[0]

        q = self.split_heads(self.q_dense(x), batch_size)
        k = self.split_heads(self.k_dense(x), batch_size)
        v = self.split_heads(self.v_dense(x), batch_size)

        matmul_qk = ops.matmul(q, ops.transpose(k, axes=[0, 1, 3, 2]))
        dk = ops.cast(self.head_dim, "float32")
        scaled_attention_logits = matmul_qk / ops.sqrt(dk)

        attention_weights = ops.softmax(scaled_attention_logits, axis=-1)
        attention_weights = self.att_drop(attention_weights, training=training)

        output = ops.matmul(attention_weights, v)
        output = ops.transpose(output, axes=[0, 2, 1, 3])
        concat_attention = ops.reshape(output, (batch_size, -1, self.emb_size))

        return self.out_dense(concat_attention)

    def get_config(self):
        config = super().get_config()
        config.update({
            "emb_size": self.emb_size,
            "num_heads": self.num_heads,
            "head_dim": self.head_dim,
        })
        return config


class TransformerEncoderBlock(layers.Layer):
    """Standard Transformer Encoder Block with Pre-LayerNormalization."""
    def __init__(self, emb_size=40, num_heads=10, forward_expansion=4, drop_p=0.5, **kwargs):
        super().__init__(**kwargs)
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.forward_expansion = forward_expansion
        self.drop_p = drop_p

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = MultiHeadSelfAttention(emb_size=emb_size, num_heads=num_heads, dropout=drop_p)
        self.drop1 = layers.Dropout(drop_p)

        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn = models.Sequential([
            layers.Dense(emb_size * forward_expansion, activation='gelu'),
            layers.Dropout(drop_p),
            layers.Dense(emb_size),
            layers.Dropout(drop_p)
        ])

    def call(self, x, training=False):
        res1 = x
        x_norm1 = self.norm1(x)
        attn_out = self.attn(x_norm1, training=training)
        x = res1 + self.drop1(attn_out, training=training)

        res2 = x
        x_norm2 = self.norm2(x)
        ffn_out = self.ffn(x_norm2, training=training)
        x = res2 + ffn_out

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "emb_size": self.emb_size,
            "num_heads": self.num_heads,
            "forward_expansion": self.forward_expansion,
            "drop_p": self.drop_p,
        })
        return config


def create_eeg_conformer(
    input_shape=(22, 1000),
    nb_classes=4,
    emb_size=40,
    depth=6,
    num_heads=10
):
    inputs = layers.Input(shape=input_shape)

    if len(input_shape) == 2:
        x = layers.Reshape((1, input_shape[0], input_shape[1]))(inputs)
    else:
        x = inputs

    x = layers.Permute((2, 3, 1))(x)

    x = layers.Conv2D(emb_size, kernel_size=(1, 25), strides=(1, 1), padding='same', use_bias=True)(x)
    n_chans = input_shape[0] if len(input_shape) == 2 else input_shape[1]
    x = layers.Conv2D(emb_size, kernel_size=(n_chans, 1), strides=(1, 1), padding='valid', use_bias=True)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)

    x = layers.AveragePooling2D(pool_size=(1, 75), strides=(1, 15))(x)
    x = layers.Dropout(0.5)(x)

    x = RearrangeToSeq()(x)

    for _ in range(depth):
        x = TransformerEncoderBlock(emb_size=emb_size, num_heads=num_heads, drop_p=0.5)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='elu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(32, activation='elu')(x)
    x = layers.Dropout(0.3)(x)

    activation = 'sigmoid' if nb_classes == 1 else 'softmax'
    outputs = layers.Dense(nb_classes, activation=activation)(x)

    model = Model(inputs=inputs, outputs=outputs, name="EEG_Conformer")
    return model


def format_eeg_tensor_conformer(data_array):
    """
    Formats input EEG data array into standard 3D matrix for EEG Conformer:
    (Batch/Epochs, Channels, Timepoints).
    """
    arr = np.asarray(data_array, dtype=np.float32)

    if arr.ndim == 2:
        return np.expand_dims(arr, axis=0)
    elif arr.ndim == 3:
        if arr.shape[1] > arr.shape[2]:  # If Time > Channels in axis 1
            return np.transpose(arr, (0, 2, 1))
        return arr
    elif arr.ndim == 4:
        arr = np.squeeze(arr)
        if arr.ndim == 2:
            return np.expand_dims(arr, axis=0)
        elif arr.shape[1] > arr.shape[2]:
            return np.transpose(arr, (0, 2, 1))
        return arr
    else:
        raise ValueError(f"Unexpected array dimension: {arr.ndim} (shape: {arr.shape})")


def run_subject_level_mc_cv_conformer(X_healthy, X_pd, SEED=42):
    X_healthy = scale_data(X_healthy)
    X_pd = scale_data(X_pd)

    # Infer input shape from single epoch: (Channels, Timepoints)
    sample_sub = X_healthy[0]
    sample_epoch = sample_sub[0] if sample_sub.ndim == 3 else sample_sub

    if sample_epoch.ndim == 2:
        if sample_epoch.shape[0] > sample_epoch.shape[1]:
            input_shape = (sample_epoch.shape[1], sample_epoch.shape[0])
        else:
            input_shape = (sample_epoch.shape[0], sample_epoch.shape[1])
    elif sample_epoch.ndim == 3:
        sample_epoch = np.squeeze(sample_epoch)
        if sample_epoch.shape[0] > sample_epoch.shape[1]:
            input_shape = (sample_epoch.shape[1], sample_epoch.shape[0])
        else:
            input_shape = (sample_epoch.shape[0], sample_epoch.shape[1])
    else:
        raise ValueError(f"Unexpected epoch shape: {sample_epoch.shape}")

    print(f"--> Inferred EEG Conformer Input Shape (Channels, Timepoints): {input_shape}")

    n_hc, n_pd = len(X_healthy), len(X_pd)
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    thresholds = list(range(65, 95, 5))

    hc_splits = list(outer_kf.split(np.arange(n_hc)))
    pd_splits = list(outer_kf.split(np.arange(n_pd)))

    total_correct = 0
    total_subjects = 0
    fold_summary_records = []

    # EEG Conformer Hyperparameter Grid Search
    param_grid = {
        'lr': [1e-3],
        'batch_size': [32],
        'emb_size': [40],
        'depth': [6],
        'num_heads': [10]
    }

    keys = param_grid.keys()
    all_combinations = [dict(zip(keys, combo)) for combo in product(*param_grid.values())]

    for fold in range(5):
        print(f"\n========================================")
        print(f"========== OUTER FOLD {fold+1} / 5 ==========")
        print(f"========================================")

        hc_train_all, hc_test = hc_splits[fold]
        pd_train_all, pd_test = pd_splits[fold]

        best_score = -1.0
        best_params = None
        best_threshold = 75

        hc_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(hc_train_all))
        pd_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(pd_train_all))

        for params in all_combinations:
            inner_fold_accuracies = []
            inner_fold_thresholds = []

            for inner_fold in range(3):
                hc_tr_in_idx, hc_val_in_idx = hc_inner_splits[inner_fold]
                pd_tr_in_idx, pd_val_in_idx = pd_inner_splits[inner_fold]

                hc_train_sub = [X_healthy[hc_train_all[i]] for i in hc_tr_in_idx]
                pd_train_sub = [X_pd[pd_train_all[i]] for i in pd_tr_in_idx]
                hc_val_sub = [X_healthy[hc_train_all[i]] for i in hc_val_in_idx]
                pd_val_sub = [X_pd[pd_train_all[i]] for i in pd_val_in_idx]

                # Balance classes for inner training
                X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_sub, pd_train_sub)
                X_inner_train = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
                y_inner_train = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)

                # Format to 3D (Batch, Channels, Timepoints)
                X_inner_train = format_eeg_tensor_conformer(X_inner_train)

                # Shuffle training data
                shuffle_idx = np.random.RandomState(SEED).permutation(len(X_inner_train))
                X_inner_train = X_inner_train[shuffle_idx]
                y_inner_train = y_inner_train[shuffle_idx]

                # Train/Val split
                val_size = int(len(X_inner_train) * 0.1)
                X_tr, y_tr = X_inner_train[val_size:], y_inner_train[val_size:]
                X_va, y_va = X_inner_train[:val_size], y_inner_train[:val_size]

                # Instantiate EEG Conformer Model
                inner_model = create_eeg_conformer(
                    input_shape=input_shape,
                    nb_classes=1,
                    emb_size=params['emb_size'],
                    depth=params['depth'],
                    num_heads=params['num_heads']
                )
                inner_model.compile(
                    optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr']),
                    loss='binary_crossentropy',
                    metrics=['accuracy']
                )

                early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
                inner_model.fit(
                    X_tr, y_tr,
                    epochs=40, batch_size=params['batch_size'],
                    verbose=1, validation_data=(X_va, y_va), callbacks=[early_stop]
                )

                # Inner validation threshold tuning
                val_subjects = hc_val_sub + pd_val_sub
                val_labels = [0] * len(hc_val_sub) + [1] * len(pd_val_sub)

                val_subject_ratios = []
                valid_val_labels = []

                for sub, true_lbl in zip(val_subjects, val_labels):
                    sub_array = format_eeg_tensor_conformer(sub)

                    if sub_array.shape[0] == 0:
                        continue

                    epoch_probs = inner_model.predict(sub_array, batch_size=params['batch_size'], verbose=0).flatten()
                    pct_pd = float(np.mean(epoch_probs) * 100)
                    val_subject_ratios.append(pct_pd)
                    valid_val_labels.append(true_lbl)

                best_t_inner, max_inner_acc = 75, -1.0
                for t in thresholds:
                    t_preds = [1 if ratio >= t else 0 for ratio in val_subject_ratios]
                    acc = accuracy_score(valid_val_labels, t_preds) if len(valid_val_labels) > 0 else 0.0
                    if acc > max_inner_acc:
                        max_inner_acc = acc
                        best_t_inner = t

                inner_fold_accuracies.append(max_inner_acc)
                inner_fold_thresholds.append(best_t_inner)

            mean_inner_acc = np.mean(inner_fold_accuracies)
            if mean_inner_acc > best_score:
                best_score = mean_inner_acc
                best_params = params
                best_threshold = int(np.median(inner_fold_thresholds))

        print(f">> Best Grid Parameters Selected: {best_params} | Threshold: {best_threshold}% (Inner Acc: {best_score:.4f})")

        # --- OUTER TRAINING & TESTING ---
        hc_train_final = [X_healthy[i] for i in hc_train_all]
        pd_train_final = [X_pd[i] for i in pd_train_all]

        X_tr_hc_final, X_tr_pd_final = balance_matrices_subject_wise(hc_train_final, pd_train_final)
        X_train_final = np.concatenate([X_tr_hc_final, X_tr_pd_final], axis=0)
        y_train_final = np.concatenate([np.zeros(len(X_tr_hc_final)), np.ones(len(X_tr_pd_final))], axis=0)

        X_train_final = format_eeg_tensor_conformer(X_train_final)

        shuffle_idx_final = np.random.RandomState(SEED).permutation(len(X_train_final))
        X_train_final = X_train_final[shuffle_idx_final]
        y_train_final = y_train_final[shuffle_idx_final]

        val_size_final = int(len(X_train_final) * 0.1)
        X_tr_f, y_tr_f = X_train_final[val_size_final:], y_train_final[val_size_final:]
        X_va_f, y_va_f = X_train_final[:val_size_final], y_train_final[:val_size_final]

        final_model = create_eeg_conformer(
            input_shape=input_shape,
            nb_classes=1,
            emb_size=best_params['emb_size'],
            depth=best_params['depth'],
            num_heads=best_params['num_heads']
        )
        final_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['lr']),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        early_stop_final = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
        final_model.fit(
            X_tr_f, y_tr_f,
            epochs=80, batch_size=best_params['batch_size'],
            verbose=1, validation_data=(X_va_f, y_va_f), callbacks=[early_stop_final]
        )

        test_subjects = [X_healthy[i] for i in hc_test] + [X_pd[i] for i in pd_test]
        test_labels = [0] * len(hc_test) + [1] * len(pd_test)
        n_hc_test = len(hc_test)
        n_pd_test = len(pd_test)

        hc_correct_count = 0
        pd_correct_count = 0

        for sub, true_label in zip(test_subjects, test_labels):
            sub_array = format_eeg_tensor_conformer(sub)

            if sub_array.shape[0] == 0:
                continue

            pct_pd = float(np.mean(final_model.predict(sub_array, batch_size=best_params['batch_size'], verbose=0).flatten()) * 100)

            vote_thresholds = [best_threshold - 5, best_threshold, best_threshold + 5]
            votes = [1 if pct_pd >= t else 0 for t in vote_thresholds]
            pred = 1 if sum(votes) >= 2 else 0

            if pred == true_label:
                if true_label == 0:
                    hc_correct_count += 1
                else:
                    pd_correct_count += 1

        fold_total_correct = hc_correct_count + pd_correct_count
        fold_total_subjects = len(test_subjects)

        total_correct += fold_total_correct
        total_subjects += fold_total_subjects

        fold_acc = (fold_total_correct / fold_total_subjects) * 100 if fold_total_subjects > 0 else 0.0

        fold_summary_records.append({
            'Fold Number': fold + 1,
            'Optimal Hyperparams': str(best_params),
            'Optimal Threshold (%)': best_threshold,
            'Healthy Correct': f"{hc_correct_count}/{n_hc_test}",
            'PD Correct': f"{pd_correct_count}/{n_pd_test}",
            'Fold Accuracy (%)': f"{fold_acc:.2f}%",
            'Total Correct': f"{fold_total_correct}/{fold_total_subjects}"
        })

        print(f"Outer Fold {fold+1} Stats -> Healthy: {hc_correct_count}/{n_hc_test} | PD: {pd_correct_count}/{n_pd_test} | Acc: {fold_acc:.2f}%")

    summary_df = pd.DataFrame(fold_summary_records)
    overall_acc = (total_correct / total_subjects) * 100 if total_subjects > 0 else 0.0

    print(f"\n========================================")
    print(f"Total Combined Correct: {total_correct}/{total_subjects}")
    print(f"Overall Nested Cross-Validation Accuracy: {overall_acc:.2f}%")
    print("\n--- Nested Cross-Validation Summary ---")
    print(summary_df.to_string(index=False))

    return summary_df

In [17]:
task = 'rest'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_mc_cv_conformer(X_hc, X_pd, SEED=42)
print(df)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_465/967291422.py:54: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_465/967291422.py:62: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_465/967291422.py:54: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_465/967291422.py:62: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.5008 - loss: 1.3661

2026-08-29 09:41:27.084207: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


157/157 ━━━━━━━━━━━━━━━━━━━━ 33s 79ms/step - accuracy: 0.5115 - loss: 1.0107 - val_accuracy: 0.5874 - val_loss: 0.6756
Epoch 2/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 72ms/step - accuracy: 0.5349 - loss: 0.7500 - val_accuracy: 0.6937 - val_loss: 0.5978
Epoch 3/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 72ms/step - accuracy: 0.6537 - loss: 0.6350 - val_accuracy: 0.7640 - val_loss: 0.5496
Epoch 4/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 72ms/step - accuracy: 0.7518 - loss: 0.5158 - val_accuracy: 0.8505 - val_loss: 0.3897
Epoch 5/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 71ms/step - accuracy: 0.8034 - loss: 0.4414 - val_accuracy: 0.8775 - val_loss: 0.3290
Epoch 6/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 71ms/step - accuracy: 0.8460 - loss: 0.3701 - val_accuracy: 0.8270 - val_loss: 0.4562
Epoch 7/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.8752 - loss: 0.3274 - val_accuracy: 0.9153 - val_loss: 0.1819
Epoch 8/80
157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 70ms/step - accuracy: 0.8926 - loss: 0.2848 - val_accurac

2026-08-29 09:54:51.458751: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-08-29 09:54:56.597420: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 3/5 | PD: 21/23 | Acc: 85.71%

Total Combined Correct: 109/144
Overall Nested Cross-Validation Accuracy: 75.69%

--- Nested Cross-Validation Summary ---
 Fold Number                                                          Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct
           1 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     80             6/6      19/24            83.33%         25/30
           2 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     65             3/6      19/23            75.86%         22/29
           3 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     65             6/6      11/23            58.62%         17/29
           4 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     65             2/5

In [18]:
task = 'walk'
band = 'alpha'
X_hc,X_pd = get_data(task, band, rest_ids, walk_ids)
df = run_subject_level_mc_cv_conformer(X_hc, X_pd, SEED=42)
print(df)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_465/967291422.py:54: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_465/967291422.py:62: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_465/967291422.py:54: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
/tmp/ipykernel_465/967291422.py:62: RuntimeWarning: The unit for channel(s) VREF has changed from V to NA.
  raw.set_channel_types(existing_mapping)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.5522 - loss: 1.6217

2026-08-29 11:15:58.569225: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


123/123 ━━━━━━━━━━━━━━━━━━━━ 28s 81ms/step - accuracy: 0.5738 - loss: 1.1536 - val_accuracy: 0.7271 - val_loss: 0.6523
Epoch 2/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 9s 71ms/step - accuracy: 0.7290 - loss: 0.5734 - val_accuracy: 0.8601 - val_loss: 0.3780
Epoch 3/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 9s 75ms/step - accuracy: 0.8186 - loss: 0.3921 - val_accuracy: 0.9060 - val_loss: 0.3775
Epoch 4/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 9s 74ms/step - accuracy: 0.8700 - loss: 0.3153 - val_accuracy: 0.9312 - val_loss: 0.2715
Epoch 5/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 9s 73ms/step - accuracy: 0.8975 - loss: 0.2500 - val_accuracy: 0.9197 - val_loss: 0.4456
Epoch 6/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 9s 73ms/step - accuracy: 0.9153 - loss: 0.2156 - val_accuracy: 0.9450 - val_loss: 0.2600
Epoch 7/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 9s 71ms/step - accuracy: 0.9310 - loss: 0.1863 - val_accuracy: 0.9472 - val_loss: 0.2104
Epoch 8/80
123/123 ━━━━━━━━━━━━━━━━━━━━ 9s 71ms/step - accuracy: 0.9415 - loss: 0.1678 - val_accuracy: 0.94

2026-08-29 11:22:17.969317: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-08-29 11:22:23.046629: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 2/4 | PD: 21/22 | Acc: 88.46%

Total Combined Correct: 106/133
Overall Nested Cross-Validation Accuracy: 79.70%

--- Nested Cross-Validation Summary ---
 Fold Number                                                          Optimal Hyperparams  Optimal Threshold (%) Healthy Correct PD Correct Fold Accuracy (%) Total Correct
           1 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     65             5/5      17/22            81.48%         22/27
           2 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     70             3/5      18/22            77.78%         21/27
           3 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     70             2/5      15/22            62.96%         17/27
           4 {'lr': 0.001, 'batch_size': 32, 'emb_size': 40, 'depth': 6, 'num_heads': 10}                     70             3/4